# Formatting and cleaning

## 1. Load basic modules and the csv file 

In [34]:
import pandas as pd
import numpy as np
import random as rd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

data = pd.read_csv('../data_raw/train.csv')
data.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


## 2. Checking Dataset Shape,  null Values and feature types

**Fixing incorrect, incomplete, duplicate or otherwise erroneous data**

In [2]:
data.shape

(381109, 12)

In [3]:
data.isnull().sum()

id                      0
Gender                  0
Age                     0
Driving_License         0
Region_Code             0
Previously_Insured      0
Vehicle_Age             0
Vehicle_Damage          0
Annual_Premium          0
Policy_Sales_Channel    0
Vintage                 0
Response                0
dtype: int64

In [4]:
data.dtypes
data.drop("id", axis = 1, inplace = True)


(381109, 11)

**As we can see, there is no null data in the dataset. We drop columns 'id' because it doesnt provide relevant information for our scope**


## 3. Importing functions from Cleaning_functions.ipynb

In [5]:
import sys
sys.path.append('../src/')
import Cleaning_functions as cf

### 3.1 Continous variables

**Removing outliers from the "Annual_Premium" column**. 

The original data are found with a large scatter, outliers are interpreted as errors in data entry. Given the distribution, it is decided to use log transformation. Log transformation  de-emphasizes outliers and allows us to potentially obtain a bell-shaped distribution. The idea is that taking the log of the data can restore symmetry to the data.

**Data with a high dispersion interpreted as erroneous are eliminated. The dataset is reduced by 80172 rows (21%).**

In [6]:
data = cf.correct_Annual_Premium(data)
data.drop("log_premium", axis = 1, inplace = True)

In [7]:
data.shape

(300937, 11)

**Changing values from the "Policy_Sales_Channel" column**

This columns has over 148 different values, we grouped into 'others' category the values that are less than 2%

In [8]:
len(data['Policy_Sales_Channel'].unique())

148

In [9]:
data['Policy_Sales_Channel'].value_counts(normalize=True)

152.0    0.377272
26.0     0.218155
124.0    0.199364
160.0    0.050143
122.0    0.028135
           ...   
43.0     0.000003
84.0     0.000003
97.0     0.000003
2.0      0.000003
99.0     0.000003
Name: Policy_Sales_Channel, Length: 148, dtype: float64

In [10]:
data = cf.correct_policy(data)



In [11]:
data.Policy_Sales_Channel.value_counts(normalize=True)

152.0    0.377272
other    0.326294
26.0     0.218155
160.0    0.050143
122.0    0.028135
Name: Policy_Sales_Channel, dtype: float64

In [12]:
data.head()

,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


In [19]:
#dropear region_code

In [ ]:
data.to_csv('../documents/data_clean.csv', index=False) #insumo para los gráficos

## Dummy variables and standardization

As we have categorical variables, we need to turn them to dummies variables

In [28]:
# ACORDARSE CAMBIAR ESTO A LAS FUNCIONES
data['Driving_License'] = data['Driving_License'].astype(str)
data['Region_Code'] = data['Region_Code'].astype(str)
data['Previously_Insured'] = data['Previously_Insured'].astype(str)


categoricals = ["Gender", "Driving_License", "Region_Code", "Previously_Insured", "Vehicle_Age", "Policy_Sales_Channel", 
               ]

enc = OneHotEncoder(drop = "first")
X = data[categoricals]
enc.fit(X)
enc.categories_

[array(['Female', 'Male'], dtype=object),
 array(['0', '1'], dtype=object),
 array(['0.0', '1.0', '10.0', '11.0', '12.0', '13.0', '14.0', '15.0',
        '16.0', '17.0', '18.0', '19.0', '2.0', '20.0', '21.0', '22.0',
        '23.0', '24.0', '25.0', '26.0', '27.0', '28.0', '29.0', '3.0',
        '30.0', '31.0', '32.0', '33.0', '34.0', '35.0', '36.0', '37.0',
        '38.0', '39.0', '4.0', '40.0', '41.0', '42.0', '43.0', '44.0',
        '45.0', '46.0', '47.0', '48.0', '49.0', '5.0', '50.0', '51.0',
        '52.0', '6.0', '7.0', '8.0', '9.0'], dtype=object),
 array(['0', '1'], dtype=object),
 array(['1-2 Year', '< 1 Year', '> 2 Years'], dtype=object),
 array(['122.0', '152.0', '160.0', '26.0', 'other'], dtype=object)]

In [29]:
dummies = enc.transform(X).toarray()

In [30]:
dummies_df = pd.DataFrame(dummies)

In [31]:
col_names = [categoricals[i] + '_' + enc.categories_[i] for i in range(len(categoricals))] 

In [32]:
col_names_drop_first = [sublist[i] for sublist in col_names for i in range(len(sublist)) if i != 0]


In [33]:
dummies_df.columns = col_names_drop_first
dummies_df.head()

,Gender_Male,Driving_License_1,Region_Code_1.0,Region_Code_10.0,Region_Code_11.0,Region_Code_12.0,Region_Code_13.0,Region_Code_14.0,Region_Code_15.0,Region_Code_16.0,...,Region_Code_7.0,Region_Code_8.0,Region_Code_9.0,Previously_Insured_1,Vehicle_Age_< 1 Year,Vehicle_Age_> 2 Years,Policy_Sales_Channel_152.0,Policy_Sales_Channel_160.0,Policy_Sales_Channel_26.0,Policy_Sales_Channel_other
0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0


In [55]:
numericals = ['Age', 'Annual_Premium', 'Vintage', 'Response']


In [74]:
data_final = pd.concat([data[numericals], dummies_df], axis = 1)

In [76]:
data_final.shape

(682046, 65)

In [67]:
X = data_final.drop('Response', axis = 1)
y = data_final.Response

In [69]:
X.isna().sum()


Age                               0
Annual_Premium                    0
Vintage                           0
Gender_Male                   80172
Driving_License_1             80172
                              ...  
Vehicle_Age_> 2 Years         80172
Policy_Sales_Channel_152.0    80172
Policy_Sales_Channel_160.0    80172
Policy_Sales_Channel_26.0     80172
Policy_Sales_Channel_other    80172
Length: 64, dtype: int64

In [68]:
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify = y, test_size = 0.2, random_state = 123)

In [61]:
X_train

,Age,Annual_Premium,Vintage,Gender_Male,Driving_License_1,Region_Code_1.0,Region_Code_10.0,Region_Code_11.0,Region_Code_12.0,Region_Code_13.0,...,Region_Code_7.0,Region_Code_8.0,Region_Code_9.0,Previously_Insured_1,Vehicle_Age_< 1 Year,Vehicle_Age_> 2 Years,Policy_Sales_Channel_152.0,Policy_Sales_Channel_160.0,Policy_Sales_Channel_26.0,Policy_Sales_Channel_other
318741,69,2630.0,68,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115282,60,2630.0,70,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
41578,37,42804.0,91,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
278301,27,43112.0,191,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
77135,46,38599.0,34,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238838,47,33905.0,34,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
306243,42,40918.0,204,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
266889,24,2630.0,72,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
230244,23,39842.0,49,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [60]:
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_val)

In [ ]:
#data.to_csv('../documents/data_models.csv', index=False) #insumo para los modelos